In [ ]:
# ==========================================================
# CELL 1: MOUNT DRIVE + CLONE/PULL REPO
# ==========================================================
from google.colab import drive
import os, sys

drive.mount('/content/drive')

REPO_URL = "https://github.com/bestoism/skripsi-corn-label-noise"
REPO_DIR = "/content/skripsi-corn-label-noise"

if os.path.exists(REPO_DIR):
    print("🔄 Repo sudah ada, menarik update terbaru...")
    !cd {REPO_DIR} && git pull
else:
    print("⬇️  Clone repo baru...")
    !git clone {REPO_URL} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n✅ Setup selesai. Working dir: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Repo sudah ada, menarik update terbaru...
Already up to date.

✅ Setup selesai. Working dir: /content/skripsi-corn-label-noise


In [ ]:
# ==========================================================
# CELL 2: INSTALL REQUIREMENTS
# ==========================================================
!pip install -q -r requirements.txt
print("✅ Dependencies terpasang.")

✅ Dependencies terpasang.


In [ ]:
# ==========================================================
# CELL 2.5 (BARU): IMPORT UMUM — dipakai di banyak cell berikutnya
# ==========================================================
import os
import pandas as pd
import numpy as np

In [ ]:
# ==========================================================
# CELL 3: SCRAPING GOOGLE PLAY STORE
# ==========================================================
# ⚠️ JALANKAN SEKALI SAJA SEUMUR PROYEK. Setelah cell ini selesai,
# all_reviews_master.csv DIKUNCI -- jangan dijalankan ulang, supaya
# seluruh eksperimen berikutnya memakai sumber data yang identik.
#
# Kalau sempat terputus di tengah jalan, JALANKAN ULANG cell ini --
# scraper akan resume otomatis dari checkpoint terakhir per (app, rating),
# bukan mulai dari nol.
# ==========================================================
from src import config

if os.path.exists(config.RAW_DATA_FILE):
    print(f"✅ {config.RAW_DATA_FILE} sudah ada -- scraping dilewati.")
    print("   Hapus file ini manual kalau memang mau scraping ulang dari nol.")
else:
    from scripts.scrape_google_play import main as run_scraping
    run_scraping()

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
 GOOGLE PLAY SCRAPER — STRATIFIED PER RATING 

📡 SeaBank | rating=1 | target=800
   +200 ulasan (total: 200/800)
   +200 ulasan (total: 400/800)
   +200 ulasan (total: 600/800)
   💾 Checkpoint tersimpan (600 baris)
   +200 ulasan (total: 800/800)
✅ Selesai: SeaBank rating=1 -> 800 ulasan bersih.

📡 SeaBank | rating=2 | target=500
   +200 ulasan (total: 200/500)
   +200 ulasan (total: 400/500)
   +200 ulasan (total: 600/500)
   💾 Checkpoint tersimpan (600 baris)
✅ Selesai: SeaBank rating=2 -> 600 ulasan bersih.

📡 SeaBank | rating=3 | target=500
   +200 ulasan (total: 200/500)
   +200 ulasan (total: 400/500)
   +200 ulasan (total: 600/500)
   💾 Checkpoint tersimpan (600 baris)
✅ Selesai: SeaBank rating=3 -> 600 ulasan bersih.

📡 SeaBank | rating=4 | target=700
   +200 ulasan (total: 200/700)
   +200 ulasan (total: 400/700)
   +200 ulasan (total: 600/700)
   💾 Checkpoint tersimpan (600 baris)


In [ ]:
# ==========================================================
# CELL 4: PREPROCESSING + TRAIN-TEST SPLIT (SUMBER KEBENARAN TUNGGAL)
# ==========================================================
# ⚠️ JALANKAN SEKALI SAJA. Split ini (train_raw / test) DIKUNCI dan dipakai
# di SELURUH eksperimen berikutnya -- termasuk pilot study proxy 0-4.
# Menjalankan ulang cell ini akan mengganti split yang sudah ada; jangan
# lakukan kecuali kamu sengaja ingin memulai dari nol.
# ==========================================================
from sklearn.model_selection import train_test_split
from src.preprocess import run_preprocessing
from src import config

if os.path.exists(config.TRAIN_RAW_FILE) and os.path.exists(config.TEST_FILE):
    print("✅ Split train/test sudah ada -- preprocessing dilewati.")
    df_train = pd.read_csv(config.TRAIN_RAW_FILE)
    df_test = pd.read_csv(config.TEST_FILE)
    print(f"   Train: {len(df_train)} baris | Test: {len(df_test)} baris")
else:
    print("=" * 60)
    print(" PREPROCESSING ")
    print("=" * 60)
    df_clean = run_preprocessing(config.RAW_DATA_FILE, config.CLEAN_TEXT_FILE)
    df_clean = df_clean.dropna(subset=["cleaned_text", "rating"])

    print("\n" + "=" * 60)
    print(" SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) ")
    print("=" * 60)
    df_train, df_test = train_test_split(
        df_clean, test_size=0.2, random_state=42, stratify=df_clean["rating"]
    )
    df_train.to_csv(config.TRAIN_RAW_FILE, index=False)
    df_test.to_csv(config.TEST_FILE, index=False)

    print(f"✅ Train: {len(df_train)} baris -> {config.TRAIN_RAW_FILE}")
    print(f"✅ Test : {len(df_test)} baris -> {config.TEST_FILE}")

⬇️  Mengunduh Kamus Alay (Salsabila dkk., 2018) dari https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv ...
✅ Tersimpan di cache: /content/drive/MyDrive/SKRIPSI_CORN/lexicon/slang_base.csv
📖 Kamus slang dasar: 15007 entri (Salsabila dkk., 2018)
 PREPROCESSING 
📥 Membaca data mentah dari: /content/drive/MyDrive/SKRIPSI_CORN/data/raw/all_reviews_master.csv
📏 Jumlah data awal: 10800 baris
🧹 Cleaning teks (lowercase, URL/tag, emoji, elongasi, slang)...
📊 Cakupan kamus slang: 14/138985 kata (0.01%)
🔍 Teks identik, rating berbeda: 162 kasus
✅ Setelah dibersihkan: 8947 baris (terbuang: 1853)
💾 Disimpan di: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/reviews_clean.csv

 SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) 
✅ Train: 7157 baris -> /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw.csv
✅ Test : 1790 baris -> /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_test.csv


In [ ]:
# ==========================================================
# CELL 5: PILOT STUDY -- ABLASI PROXY 0-4
# ==========================================================
# Menjalankan clean.py untuk KELIMA metode proxy secara berurutan.
# Hasilnya terkumpul otomatis di results/proxy_ablation_table.csv
# (dipakai untuk narasi pilot study di Bab 1).
#
# CATATAN: proxy 2, 3 butuh fine-tuning K-Fold (5 fold x beberapa epoch),
# jadi cell ini bisa makan waktu cukup lama untuk kelimanya. Kalau kamu
# sudah yakin final proxy = 3 (finetuned_corn) dan hanya ingin lihat
# pilot study SEKALI dan sudah tahu hasilnya, cell ini boleh dilewati --
# langsung ke Cell 6.
#
# PROXY_ID 4 (fusion) akan raise NotImplementedError -- beri tahu saya
# kalau kamu mau lanjut mengimplementasikannya nanti.
# ==========================================================
import pandas as pd
from src import config
from src.clean import run_confident_learning

PILOT_PROXY_IDS = [0, 1, 2, 3]  # 4 (fusion) belum diimplementasikan penuh

for pid in PILOT_PROXY_IDS:
    print(f"\n{'='*70}\n PILOT STUDY -- PROXY_ID = {pid} \n{'='*70}")

    config.set_proxy(pid)

    try:
        run_confident_learning()
    except Exception as e:
        print(f"⚠️ Proxy {pid} gagal: {e}")
        continue

print("\n✅ Pilot study selesai.")
pilot_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
display(pilot_table)


 PILOT STUDY -- PROXY_ID = 0 
📌 Proxy aktif: [0] frozen_cls_lr — CLS embedding beku + Logistic Regression (P1)
 CONFIDENT LEARNING — proxy aktif: [0] frozen_cls_lr 
📥 Memuat 7157 baris data train.

🧮 Menghitung OOF pred_probs — proxy [0] frozen_cls_lr


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding (cls): 100%|██████████| 448/448 [00:28<00:00, 15.84it/s]



📐 Kualitas proxy [frozen_cls_lr]:
   Exact Accuracy : 0.4234
   MAE            : 0.9462
   Off-by-1 Acc   : 0.7486
   QWK            : 0.6103

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3495 baris diflag (48.83%)
   'prune_by_noise_rate': 2980 baris diflag (41.64%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3495 / sisa 3662
   Severity-aware : buang 1467 / sisa 5690

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2028
2     928
3     432
4     107
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_hard__frozen_cls_lr.csv
💾 Cleaned (severe) -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_severe__frozen_cls_lr.csv

📝 Sample validasi manusia (50 baris, stratified by rating_diff):
   /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5000
      Epoch 2/3 - Loss: 0.4358
      Epoch 3/3 - Loss: 0.3685
   [Proxy finetuned_corn] Fold 5/5 (train=5726, val=1431)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5074
      Epoch 2/3 - Loss: 0.4416
      Epoch 3/3 - Loss: 0.3823

📐 Kualitas proxy [finetuned_corn]:
   Exact Accuracy : 0.4394
   MAE            : 0.8041
   Off-by-1 Acc   : 0.8223
   QWK            : 0.6692

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3649 baris diflag (50.99%)
   'prune_by_noise_rate': 3288 baris diflag (45.94%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3649 / sisa 3508
   Severity-aware : buang 1145 / sisa 6012

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2504
2     838
3     239
4      68
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_hard__finetuned_corn.csv
💾 Cleaned (severe) -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_severe__finetuned_corn.csv

📝 Sample validasi manusia (50 baris, stra

,proxy_id,proxy_name,proxy_desc,accuracy,mae,off_by_one,qwk,pct_flagged_noise
0,0,frozen_cls_lr,CLS embedding beku + Logistic Regression (P1),0.423362,0.946207,0.748638,0.610310,48.833310
1,1,frozen_meanpool_lr,Mean-pooling embedding beku + Logistic Regress...,0.422384,0.953752,0.748777,0.598690,47.953053
2,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",0.438731,0.801593,0.817242,0.653276,51.893251
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.439430,0.804108,0.822272,0.669240,50.985050


In [ ]:
# ==========================================================
# CELL 6: PROXY FINAL (SESUAI BAB 3) -- HASIL INI YANG DIPAKAI BAB 4
# ==========================================================
# ⚠️ PENTING: restart runtime dulu sebelum cell ini kalau tadi sempat
# jalankan Cell 5 (loop pilot study) -- supaya config bersih, tidak ada
# sisa reload yang bikin path tercampur.
# ==========================================================
from src import config

config.set_proxy(3)
print(f"📌 Proxy final terkunci: [{config.PROXY_ID}] {config.PROXY_NAME}")

from src.clean import run_confident_learning
df_noise, proxy_metrics = run_confident_learning()

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
📌 Proxy final terkunci: [3] finetuned_corn
 CONFIDENT LEARNING — proxy aktif: [3] finetuned_corn 
📥 Memuat 7157 baris data train.

🧮 Menghitung OOF pred_probs — proxy [3] finetuned_corn
⚡ Memuat cache proxy [finetuned_corn] ...

📐 Kualitas proxy [finetuned_corn]:
   Exact Accuracy : 0.4394
   MAE            : 0.8041
   Off-by-1 Acc   : 0.8223
   QWK            : 0.6692

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3649 baris diflag (50.99%)
   'prune_by_noise_rate': 3288 baris diflag (45.94%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3649 / sisa 3508
   Severity-aware : buang 1145 / sisa 6012

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2504
2     838
3     239
4      68
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /

In [ ]:
# Cell pengecekan cepat -- jalankan SETELAH Cell 1, 2, 2.5, SEBELUM Cell 6.5
from src import config
import os

checks = {
    "Data train": config.TRAIN_RAW_FILE,
    "Data test": config.TEST_FILE,
    "Tabel ablasi proxy": config.PROXY_QUALITY_LOG_FILE,
    "Hasil 6 skenario": config.FINAL_RESULTS_TABLE_FILE,
}

all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"{status} {label}: {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ Semua file ditemukan -- Drive terhubung dengan benar, aman lanjut ke Cell 6.5.")
else:
    print("\n⛔ Ada file tidak ditemukan! Cek apakah kamu login Drive dengan akun yang BENAR")
    print("   (akun yang sama yang dipakai untuk membuat folder SKRIPSI_CORN).")

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
✅ Data train: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw.csv
✅ Data test: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_test.csv
✅ Tabel ablasi proxy: /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv
✅ Hasil 6 skenario: /content/drive/MyDrive/SKRIPSI_CORN/results/final_results_table.csv

✅ Semua file ditemukan -- Drive terhubung dengan benar, aman lanjut ke Cell 6.5.


In [ ]:
# ==========================================================
# CELL 6.5 (BARU): ABLASI TAMBAHAN -- P5 (FUSION SENTIMEN)
# ==========================================================
# P5 BUKAN bagian dari pilot study (Cell 5) dan BUKAN proxy final (Cell 6) --
# ini ablasi tambahan sesuai Tabel 3.2 Bab 3, hasilnya untuk Bab 4 saja.
# Menambah satu baris ke proxy_ablation_table.csv (proxy_id=4), TIDAK
# mempengaruhi hasil training M1-M6 yang sudah selesai (semua pakai proxy 3).
#
# ⚠️ PENTING: cell ini WAJIB diakhiri dengan set_proxy(3) lagi -- supaya
# proxy aktif kembali ke final (3) sebelum cell manapun setelah ini jalan.
# ==========================================================
import traceback
from src import config
from src.clean import run_confident_learning

config.set_proxy(4)
try:
    df_noise_p5, proxy_metrics_p5 = run_confident_learning()
except Exception as e:
    print(f"⚠️ Proxy 4 (fusion) gagal:")
    traceback.print_exc()   # <-- ini yang penting, tunjukkan baris persisnya
finally:
    config.set_proxy(3)
    print(f"\n📌 Proxy dikembalikan ke final: [{config.PROXY_ID}] {config.PROXY_NAME}")

📌 Proxy aktif: [4] finetuned_corn_fusion — IndoBERT+CORN + fusi sentimen eksternal (P5)
 CONFIDENT LEARNING — proxy aktif: [4] finetuned_corn_fusion 
📥 Memuat 7157 baris data train.

🧮 Menghitung OOF pred_probs — proxy [4] finetuned_corn_fusion
⬇️  Memuat model sentimen: w11wo/indonesian-roberta-base-sentiment-classifier


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

   Jumlah kelas sentimen model ini: 3 ({0: 'positive', 1: 'neutral', 2: 'negative'})


Sentiment scoring: 100%|██████████| 224/224 [00:29<00:00,  7.48it/s]


   [Proxy finetuned_corn_fusion] Fold 1/5 (train=5725, val=1432)...


...

      Epoch 1/3 - Loss: 0.5025
      Epoch 2/3 - Loss: 0.4360
      Epoch 3/3 - Loss: 0.3704
   [Proxy finetuned_corn_fusion] Fold 4/5 (train=5726, val=1431)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5021
      Epoch 2/3 - Loss: 0.4392
      Epoch 3/3 - Loss: 0.3730
   [Proxy finetuned_corn_fusion] Fold 5/5 (train=5726, val=1431)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5075
      Epoch 2/3 - Loss: 0.4452
      Epoch 3/3 - Loss: 0.3799

📐 Kualitas proxy [finetuned_corn_fusion]:
   Exact Accuracy : 0.4467
   MAE            : 0.8005
   Off-by-1 Acc   : 0.8153
   QWK            : 0.6685

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3677 baris diflag (51.38%)
   'prune_by_noise_rate': 3248 baris diflag (45.38%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3677 / sisa 3480
   Severity-aware : buang 1246 / sisa 5911

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2431
2     928
3     245
4      73
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_hard__finetuned_corn_fusion.csv
💾 Cleaned (severe) -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_severe__finetuned_corn_fusion.csv

📝 Sample validasi ma

In [ ]:
# ==========================================================
# CELL 7: VALIDASI MANUSIA — WAJIB SEBELUM LANJUT
# ==========================================================
from src import config

print("⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 8 ⚠️")
print(f"1. Buka: {config.HUMAN_VALIDATION_FILE}")
print("2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:")
print("   'noise' / 'not_noise' / 'ambiguous'")
print("3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).")
print("4. Jalankan cell ini lagi untuk cek kelengkapan + hitung agreement rate.")

⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 8 ⚠️
1. Buka: /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_sample.csv
2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:
   'noise' / 'not_noise' / 'ambiguous'
3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).
4. Jalankan cell ini lagi untuk cek kelengkapan + hitung agreement rate.


In [ ]:
from src import config

with open(config.HUMAN_VALIDATION_FILE, encoding="utf-8") as f:
    lines = f.readlines()

print(f"Total baris (termasuk header): {len(lines)}")
print("\n--- Header (baris 1) ---")
print(repr(lines[0]))
print("\n--- Baris 9-13 (sekitar baris error) ---")
for i in range(8, 13):
    if i < len(lines):
        print(f"[baris {i+1}] {repr(lines[i])}")

Total baris (termasuk header): 51

--- Header (baris 1) ---
'source_app;review_text;cleaned_text;rating;predicted_rating;rating_diff;human_verdict;human_note\n'

--- Baris 9-13 (sekitar baris error) ---
[baris 9] 'SeaBank;kurang cepat;kurang cepat;4;2;2;ambiguous;Hanya menyebut kurang cepat tanpa konteks tingkat masalah\n'
[baris 10] 'Gojek;mobil nya bersih wangi,dan pelayanannya ramah;mobil nya bersih wangi,dan pelayanannya ramah;4;5;1;not_noise;Ulasan positif jelas dan langsung mendukung rating 4.\n'
[baris 11] 'Tokopedia;Kenapa skrg mau beli obat di tokped ribet banget yah, harus konsultasi dokter & toko obat nya pun di tentukan oleh tokped..sering ketemu yg stok obat nya kosong, jd harus chat dokter ulang..cape deh..;kenapa skrg mau beli obat di tokped ribet banget yah, harus konsultasi dokter & toko obat nya pun di tentukan oleh tokped.sering ketemu yg stok obat nya kosong, jd harus chat dokter ulang.cape deh.;3;2;1;not_noise;Ulasan berisi keluhan nyata tetapi juga konteks penggun

In [ ]:
# Jalankan sel ini SETELAH selesai isi manual di atas.
from src.human_validation import compute_agreement

result = compute_agreement()
if result is not None:
    print("\n✅ Validasi manusia lengkap. Siap lanjut ke Cell 8 (training).")
else:
    print("\n⛔ Belum lengkap/ada error -- perbaiki dulu sebelum lanjut.")

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
   (file dibaca dengan delimiter ';')
 HASIL VALIDASI MANUSIA vs CLEANLAB 
Total sample direview : 50
Setuju (memang noise) : 6 (12.0%)
Ambigu                : 3 (6.0%)
Tidak setuju          : 41 (82.0%)

📊 Agreement rate per rating_diff (mendukung/menolak asumsi severity-aware pruning):
   diff=1: n=34 | noise=0.0% | not_noise=94.1% | ambiguous=5.9%
   diff=2: n=12 | noise=16.7% | not_noise=75.0% | ambiguous=8.3%
   diff=3: n=2 | noise=100.0% | not_noise=0.0% | ambiguous=0.0%
   diff=4: n=2 | noise=100.0% | not_noise=0.0% | ambiguous=0.0%
------------------------------------------------------------
Acuan pembanding (Northcutt et al., 2021, ImageNet): ~58% sample
yang direview terbukti benar-benar issue -- ini acuan wajar, bukan
standar mutlak yang harus dicapai (skala dan domain berbeda).

💾 Hasil lengkap -> /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_result.csv

✅

In [ ]:
# ==========================================================
# CELL 8: TRAINING 6 SKENARIO x 3 SEED (AUTO-RESUME)
# ==========================================================
import json
import numpy as np
from src.train import run_experiment
from src import config

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_FILE,           "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_FILE,           "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "corn"},
]

if os.path.exists(config.PROGRESS_FILE):
    with open(config.PROGRESS_FILE) as f:
        saved_progress = json.load(f)
    print("🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...")
else:
    saved_progress = {}
    print("🆕 Memulai training dari awal...")

all_results = []
print(f"\n🔥 {len(scenarios)} SKENARIO x {len(config.SEED_LIST)} SEED 🔥\n")

for scenario in scenarios:
    name = scenario["name"]
    metrics_list = {"mae": [], "rmse": [], "accuracy": [], "off_by_one": [], "qwk": []}
    saved_progress.setdefault(name, {})

    for seed in config.SEED_LIST:
        seed_key = str(seed)
        if seed_key in saved_progress[name]:
            print(f"⏩ {name} | seed {seed} (sudah selesai)")
            metrics = saved_progress[name][seed_key]
        else:
            metrics = run_experiment(name, scenario["train_path"], scenario["loss"], seed)
            saved_progress[name][seed_key] = metrics
            with open(config.PROGRESS_FILE, "w") as f:
                json.dump(saved_progress, f, indent=2)

        for k in metrics_list:
            metrics_list[k].append(metrics[k])

    all_results.append({
        "Model": name,
        "MAE (↓)": f"{np.mean(metrics_list['mae']):.4f} ± {np.std(metrics_list['mae']):.4f}",
        "RMSE (↓)": f"{np.mean(metrics_list['rmse']):.4f} ± {np.std(metrics_list['rmse']):.4f}",
        "Acc (↑)": f"{np.mean(metrics_list['accuracy']):.4f} ± {np.std(metrics_list['accuracy']):.4f}",
        "Off-by-1 (↑)": f"{np.mean(metrics_list['off_by_one']):.4f} ± {np.std(metrics_list['off_by_one']):.4f}",
        "QWK (↑)": f"{np.mean(metrics_list['qwk']):.4f} ± {np.std(metrics_list['qwk']):.4f}",
        "_raw_mae": np.mean(metrics_list["mae"]),
    })

df_final = pd.DataFrame(sorted(all_results, key=lambda x: x["_raw_mae"])).drop(columns=["_raw_mae"])
df_final.to_csv(config.FINAL_RESULTS_TABLE_FILE, index=False)

print("\n" + "=" * 100)
print(" 🏆 HASIL 6 SKENARIO (untuk Bab 4) 🏆")
print("=" * 100)
display(df_final)

🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...

🔥 6 SKENARIO x 3 SEED 🔥

⏩ M1_Baseline_CE | seed 42 (sudah selesai)
⏩ M1_Baseline_CE | seed 123 (sudah selesai)
⏩ M1_Baseline_CE | seed 2024 (sudah selesai)

🚀 M2_CleanedHard_CE | Seed: 42 | Loss: CE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/content/skripsi-corn-label-noise/src/train.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/content/skripsi-corn-label-noise/src/train.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/8 - Loss: 0.8883 - Val MAE: 0.3191 | QWK: 0.9237 | Off-by-1: 0.9630 | Acc: 0.7208


/content/skripsi-corn-label-noise/src/train.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


...

/content/skripsi-corn-label-noise/src/train.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 4/8 - Loss: 0.2210 - Val MAE: 0.5183 | QWK: 0.8323 | Off-by-1: 0.9269 | Acc: 0.5681


/content/skripsi-corn-label-noise/src/train.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 5/8 - Loss: 0.1678 - Val MAE: 0.5365 | QWK: 0.8376 | Off-by-1: 0.9419 | Acc: 0.5349
   ⏹ Early stopping di epoch 5 (Val MAE tidak membaik 3x)
🏆 Test (dari model Val MAE terbaik=0.5183): MAE=0.7547 | RMSE=1.1144 | Acc=0.4436 | Off-by-1=0.8419 | QWK=0.7056

 🏆 HASIL 6 SKENARIO (untuk Bab 4) 🏆


,Model,MAE (↓),RMSE (↓),Acc (↑),Off-by-1 (↑),QWK (↑)
0,M5_CleanedHard_CORN,0.7493 ± 0.0009,1.1136 ± 0.0039,0.4466 ± 0.0030,0.8456 ± 0.0018,0.7099 ± 0.0025
1,M1_Baseline_CE,0.7585 ± 0.0079,1.1270 ± 0.0135,0.4423 ± 0.0177,0.8469 ± 0.0082,0.6973 ± 0.0108
2,M6_CleanedSevere_CORN,0.7588 ± 0.0029,1.1112 ± 0.0024,0.4315 ± 0.0087,0.8505 ± 0.0062,0.7017 ± 0.0041
3,M4_Baseline_CORN,0.7637 ± 0.0051,1.1238 ± 0.0111,0.4354 ± 0.0005,0.8445 ± 0.0011,0.7056 ± 0.0074
4,M2_CleanedHard_CE,0.7655 ± 0.0048,1.1298 ± 0.0072,0.4367 ± 0.0107,0.8421 ± 0.0074,0.6985 ± 0.0021
5,M3_CleanedSevere_CE,0.7696 ± 0.0106,1.1346 ± 0.0189,0.4367 ± 0.0011,0.8395 ± 0.0088,0.6975 ± 0.0045


In [ ]:
# ==========================================================
# CELL 9: UJI SIGNIFIKANSI -- WILCOXON + HOLM-BONFERRONI
# ==========================================================
# Berbeda dari 4 percobaan sebelumnya: prediksi dikumpulkan dari
# KETIGA seed (bukan cuma seed 42), diagregasi dulu, baru diuji --
# sesuai janji Subbab 3.9.2 proposal.
# ==========================================================
from src.significance import (
    collect_all_predictions,
    aggregate_errors_across_seeds,
    run_significance_test,
    run_all_effect_sizes,
)

print("📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...")
true_labels, preds_per_seed = collect_all_predictions(scenarios)

print("\n📊 Mengagregasi absolute error across seed...")
aggregated_errors = aggregate_errors_across_seeds(true_labels, preds_per_seed)

print("\n🔬 Uji Wilcoxon Signed-Rank (3 hipotesis pre-registered) + Holm-Bonferroni...")
sig_results = run_significance_test(aggregated_errors)
display(sig_results)

print("\n📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...")
effect_sizes = run_all_effect_sizes(aggregated_errors)
display(effect_sizes)

📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...
   Memuat prediksi M1_Baseline_CE | seed 42 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Memuat prediksi M1_Baseline_CE | seed 123 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Memuat prediksi M1_Baseline_CE | seed 2024 ...


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


📊 Mengagregasi absolute error across seed...

🔬 Uji Wilcoxon Signed-Rank (3 hipotesis pre-registered) + Holm-Bonferroni...

💾 Hasil uji signifikansi -> /content/drive/MyDrive/SKRIPSI_CORN/results/significance_test.csv


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.763687,0.758473,0.506134,1.000000,Tidak
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.758845,0.763687,0.614479,1.000000,Tidak
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.749348,0.763687,0.294322,0.882967,Tidak



📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...


,model_a,model_b,mean_diff,ci_95_low,ci_95_high
0,M4_Baseline_CORN,M1_Baseline_CE,0.005214,-0.015829,0.026820
1,M6_CleanedSevere_CORN,M4_Baseline_CORN,-0.004842,-0.024395,0.014339
2,M5_CleanedHard_CORN,M4_Baseline_CORN,-0.014339,-0.032779,0.004097


In [ ]:
# ==========================================================
# CELL 10: RINGKASAN AKHIR -- SIAP DISALIN KE BAB 4
# ==========================================================
# Semua dibaca dari file di Drive (bukan variabel session) -- supaya cell
# ini bisa dijalankan kapan saja, tidak peduli cell mana yang sudah/belum
# jalan di sesi Colab saat ini.
# ==========================================================
import pandas as pd
from src import config

print("=" * 70)
print(" RINGKASAN LENGKAP UNTUK BAB 4 ")
print("=" * 70)

print("\n[1] TABEL ABLASI PROXY (P0-P4) -- untuk Bab 1 (pilot study):")
if os.path.exists(config.PROXY_QUALITY_LOG_FILE):
    proxy_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
    display(proxy_table)

    print(f"\n[2] KUALITAS PROXY FINAL [{config.PROXY_NAME}] -- untuk Bab 4:")
    final_proxy_row = proxy_table[proxy_table["proxy_id"] == config.PROXY_ID]
    if not final_proxy_row.empty:
        display(final_proxy_row)
    else:
        print(f"   ⚠️ Proxy_id={config.PROXY_ID} belum ada di tabel -- jalankan Cell 6 dulu.")
else:
    print("   ⚠️ Belum ada -- jalankan Cell 5 atau 6 dulu.")

print("\n[3] VALIDASI MANUSIA -- untuk Bab 4:")
if os.path.exists(config.HUMAN_VALIDATION_RESULT_FILE):
    df_human = pd.read_csv(config.HUMAN_VALIDATION_RESULT_FILE)
    counts = df_human["human_verdict"].value_counts()
    total = len(df_human)
    print(f"   Total: {total} | Noise: {counts.get('noise',0)} ({counts.get('noise',0)/total*100:.1f}%) "
          f"| Not noise: {counts.get('not_noise',0)} ({counts.get('not_noise',0)/total*100:.1f}%) "
          f"| Ambiguous: {counts.get('ambiguous',0)} ({counts.get('ambiguous',0)/total*100:.1f}%)")
else:
    print("   ⚠️ Belum ada -- jalankan Cell 7 (compute_agreement) dulu.")

print("\n[4] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:")
if os.path.exists(config.FINAL_RESULTS_TABLE_FILE):
    display(pd.read_csv(config.FINAL_RESULTS_TABLE_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 8 dulu.")

print("\n[5] UJI SIGNIFIKANSI (3 hipotesis pre-registered) -- untuk Bab 4:")
if os.path.exists(config.SIGNIFICANCE_TEST_FILE):
    display(pd.read_csv(config.SIGNIFICANCE_TEST_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 9 dulu.")

print("\n[6] EFFECT SIZE + CI 95% -- untuk Bab 4:")
effect_size_file = os.path.join(config.RESULTS_DIR, "effect_sizes.csv")
if os.path.exists(effect_size_file):
    display(pd.read_csv(effect_size_file))
else:
    print("   ⚠️ Belum tersimpan ke file -- lihat catatan di bawah.")

print("\n✅ Semua file hasil tersimpan di:", config.RESULTS_DIR)

 RINGKASAN LENGKAP UNTUK BAB 4 

[1] TABEL ABLASI PROXY (P0-P4) -- untuk Bab 1 (pilot study):


,proxy_id,proxy_name,proxy_desc,accuracy,mae,off_by_one,qwk,pct_flagged_noise
0,0,frozen_cls_lr,CLS embedding beku + Logistic Regression (P1),0.423362,0.946207,0.748638,0.610310,48.833310
1,1,frozen_meanpool_lr,Mean-pooling embedding beku + Logistic Regress...,0.422384,0.953752,0.748777,0.598690,47.953053
2,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",0.438731,0.801593,0.817242,0.653276,51.893251
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.439430,0.804108,0.822272,0.669240,50.985050



[2] KUALITAS PROXY FINAL [finetuned_corn] -- untuk Bab 4:


,proxy_id,proxy_name,proxy_desc,accuracy,mae,off_by_one,qwk,pct_flagged_noise
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.43943,0.804108,0.822272,0.66924,50.98505



[3] VALIDASI MANUSIA -- untuk Bab 4:
   Total: 50 | Noise: 6 (12.0%) | Not noise: 41 (82.0%) | Ambiguous: 3 (6.0%)

[4] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:


,Model,MAE (↓),RMSE (↓),Acc (↑),Off-by-1 (↑),QWK (↑)
0,M5_CleanedHard_CORN,0.7493 ± 0.0009,1.1136 ± 0.0039,0.4466 ± 0.0030,0.8456 ± 0.0018,0.7099 ± 0.0025
1,M1_Baseline_CE,0.7585 ± 0.0079,1.1270 ± 0.0135,0.4423 ± 0.0177,0.8469 ± 0.0082,0.6973 ± 0.0108
2,M6_CleanedSevere_CORN,0.7588 ± 0.0029,1.1112 ± 0.0024,0.4315 ± 0.0087,0.8505 ± 0.0062,0.7017 ± 0.0041
3,M4_Baseline_CORN,0.7637 ± 0.0051,1.1238 ± 0.0111,0.4354 ± 0.0005,0.8445 ± 0.0011,0.7056 ± 0.0074
4,M2_CleanedHard_CE,0.7655 ± 0.0048,1.1298 ± 0.0072,0.4367 ± 0.0107,0.8421 ± 0.0074,0.6985 ± 0.0021
5,M3_CleanedSevere_CE,0.7696 ± 0.0106,1.1346 ± 0.0189,0.4367 ± 0.0011,0.8395 ± 0.0088,0.6975 ± 0.0045



[5] UJI SIGNIFIKANSI (3 hipotesis pre-registered) -- untuk Bab 4:


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.763687,0.758473,0.506134,1.000000,Tidak
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.758845,0.763687,0.614479,1.000000,Tidak
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.749348,0.763687,0.294322,0.882967,Tidak



[6] EFFECT SIZE + CI 95% -- untuk Bab 4:
   ⚠️ Belum tersimpan ke file -- lihat catatan di bawah.

✅ Semua file hasil tersimpan di: /content/drive/MyDrive/SKRIPSI_CORN/results
